# 🌾 Krishi Mitra: 20-Class Plant Disease Deep Learning Classifier
### Final Year TARP Project • Vision-Based Agricultural Extension Agent

This Google Colab notebook trains an optimized **MobileNetV3** transfer-learning model to detect **20 primary plant leaf diseases and conditions** across major crops (**Tomato, Potato, Corn/Maize, Apple, Grape**).

---
### ⚙️ How to Run in Google Colab:
1. In the Colab menu, go to **Runtime** ➔ **Change runtime type** ➔ Select **T4 GPU** (Free tier).
2. Click **Runtime** ➔ **Run all** (or press `Ctrl + F9`).
3. Training takes **~6–8 minutes** on T4 GPU.
4. When finished, the notebook automatically downloads `krishimitra_disease_model.zip` containing:
   - `plant_disease_model.pth` (~11 MB)
   - `plant_disease_model.onnx`
   - `class_indices.json`
   - `disease_remedies.json`
5. Unzip and copy these files into your local `krishimitra/model/` directory!

In [ ]:
# Step 1: Verify GPU Environment & Install Dependencies
import torch
import torchvision
import os
import sys

print(f"PyTorch Version: {torch.__version__}")
print(f"Torchvision Version: {torchvision.__version__}")

if torch.cuda.is_available():
    device = torch.device('cuda')
    print(f"✅ GPU Accelerated Environment Active: {torch.cuda.get_device_name(0)}")
else:
    device = torch.device('cpu')
    print("⚠️ GPU not detected! Please go to Runtime -> Change runtime type -> Select T4 GPU.")

!pip install -q kagglehub onnx


In [ ]:
# Step 2: Download Plant Disease Dataset & Filter 20 Target Classes
import kagglehub
import shutil
from pathlib import Path

print("📥 Downloading Plant Disease dataset via kagglehub...")
# Downloads high-resolution plant pathology dataset (publicly accessible)
dataset_path = kagglehub.dataset_download("vipoooool/new-plant-diseases-dataset")
print(f"Dataset downloaded to: {dataset_path}")

# Define the 20 primary agricultural disease classes for Krishi Mitra
TARGET_20_CLASSES = [
    # Apple
    "Apple___Apple_scab",
    "Apple___Black_rot",
    "Apple___healthy",
    # Corn (Maize)
    "Corn_(maize)___Common_rust_",
    "Corn_(maize)___Northern_Leaf_Blight",
    "Corn_(maize)___healthy",
    # Grape
    "Grape___Black_rot",
    "Grape___Esca_(Black_Measles)",
    "Grape___healthy",
    # Potato
    "Potato___Early_blight",
    "Potato___Late_blight",
    "Potato___healthy",
    # Tomato (Major Indian Vegetable Crop)
    "Tomato___Bacterial_spot",
    "Tomato___Early_blight",
    "Tomato___Late_blight",
    "Tomato___Leaf_Mold",
    "Tomato___Septoria_leaf_spot",
    "Tomato___Target_Spot",
    "Tomato___Yellow_Leaf_Curl_Virus",
    "Tomato___healthy"
]

print(f"Targeting {len(TARGET_20_CLASSES)} agricultural classes:")
for i, c in enumerate(TARGET_20_CLASSES, 1):
    print(f"  {i:2d}. {c}")


In [ ]:
# Step 3: Create Clean Curated Train/Val Split for the 20 Classes
import os
from glob import glob

base_dir = Path(dataset_path)
# Locate train and valid folders
train_candidates = list(base_dir.rglob("*train*"))
valid_candidates = list(base_dir.rglob("*valid*"))

train_source = None
for p in train_candidates:
    if p.is_dir() and any((p / c).exists() for c in TARGET_20_CLASSES):
        train_source = p
        break

valid_source = None
for p in valid_candidates:
    if p.is_dir() and any((p / c).exists() for c in TARGET_20_CLASSES):
        valid_source = p
        break

print(f"Source Train dir: {train_source}")
print(f"Source Valid dir: {valid_source}")

# Setup local working directory in Colab
curated_dir = Path("/content/krishimitra_20_data")
curated_train = curated_dir / "train"
curated_val = curated_dir / "val"

os.makedirs(curated_train, exist_ok=True)
os.makedirs(curated_val, exist_ok=True)

# Link or copy only the 20 target classes
for class_name in TARGET_20_CLASSES:
    src_tr = train_source / class_name
    src_val = valid_source / class_name
    
    dst_tr = curated_train / class_name
    dst_val = curated_val / class_name
    
    if not dst_tr.exists() and src_tr.exists():
        os.symlink(src_tr, dst_tr)
    if not dst_val.exists() and src_val.exists():
        os.symlink(src_val, dst_val)

train_count = sum(len(list((curated_train / c).glob("*"))) for c in TARGET_20_CLASSES if (curated_train / c).exists())
val_count = sum(len(list((curated_val / c).glob("*"))) for c in TARGET_20_CLASSES if (curated_val / c).exists())

print(f"✅ Curated Dataset Ready!")
print(f"   Training Images: {train_count}")
print(f"   Validation Images: {val_count}")


In [ ]:
# Step 4: Data Augmentation & PyTorch DataLoaders
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

IMG_SIZE = 224
BATCH_SIZE = 64

# Field-realistic agricultural data augmentations
train_transforms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.3),
    transforms.RandomRotation(degrees=20),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_transforms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

train_dataset = datasets.ImageFolder(str(curated_train), transform=train_transforms)
val_dataset = datasets.ImageFolder(str(curated_val), transform=val_transforms)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

class_names = train_dataset.classes
print(f"Classes mapped ({len(class_names)}): {class_names}")


In [ ]:
# Step 5: Define MobileNetV3 Transfer Learning Model
import torch.nn as nn
from torchvision.models import mobilenet_v3_small, MobileNet_V3_Small_Weights

print("Loading pre-trained MobileNetV3-Small backbone (ImageNet weights)...")
model = mobilenet_v3_small(weights=MobileNet_V3_Small_Weights.DEFAULT)

# Fine-tune classifier head for 20 classes
num_features = model.classifier[3].in_features
model.classifier[3] = nn.Sequential(
    nn.Linear(num_features, 256),
    nn.Hardswish(),
    nn.Dropout(p=0.3),
    nn.Linear(256, len(class_names))
)

model = model.to(device)

# Loss function with label smoothing to prevent overfitting on lab images
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

# Optimizer & Cosine Annealing Learning Rate Scheduler
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=10, eta_min=1e-6)

print(model.classifier)
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total trainable parameters: {trainable_params:,}")


In [ ]:
# Step 6: Execute Training Loop (10 Epochs on GPU)
import time
from tqdm.auto import tqdm

EPOCHS = 10
best_val_acc = 0.0
best_model_path = "plant_disease_model.pth"

history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}

print("🚀 Starting Training on GPU...")
start_time = time.time()

for epoch in range(1, EPOCHS + 1):
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    
    for images, labels in tqdm(train_loader, desc=f"Epoch {epoch}/{EPOCHS} [Train]"):
        images, labels = images.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item() * images.size(0)
        _, preds = torch.max(outputs, 1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)
        
    scheduler.step()
    
    epoch_train_loss = running_loss / total
    epoch_train_acc = correct / total
    
    # Validation phase
    model.eval()
    val_loss, val_correct, val_total = 0.0, 0, 0
    
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            
            val_loss += loss.item() * images.size(0)
            _, preds = torch.max(outputs, 1)
            val_correct += (preds == labels).sum().item()
            val_total += labels.size(0)
            
    epoch_val_loss = val_loss / val_total
    epoch_val_acc = val_correct / val_total
    
    history['train_loss'].append(epoch_train_loss)
    history['train_acc'].append(epoch_train_acc)
    history['val_loss'].append(epoch_val_loss)
    history['val_acc'].append(epoch_val_acc)
    
    print(f"Epoch {epoch:02d}/{EPOCHS:02d} | "
          f"Train Loss: {epoch_train_loss:.4f} - Train Acc: {epoch_train_acc*100:.2f}% | "
          f"Val Loss: {epoch_val_loss:.4f} - Val Acc: {epoch_val_acc*100:.2f}%")
          
    if epoch_val_acc > best_val_acc:
        best_val_acc = epoch_val_acc
        torch.save(model.state_dict(), best_model_path)
        print(f"   ⭐ New Best Model Saved! Validation Accuracy: {best_val_acc*100:.2f}%")

elapsed = time.time() - start_time
print(f"\n🎉 Training Complete in {elapsed/60:.2f} minutes! Best Val Accuracy: {best_val_acc*100:.2f}%")


In [ ]:
# Step 7: Plot Accuracy Curves and Classification Metrics
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(range(1, EPOCHS + 1), history['train_acc'], label='Train Accuracy', marker='o')
plt.plot(range(1, EPOCHS + 1), history['val_acc'], label='Val Accuracy', marker='s')
plt.title('Training & Validation Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
plt.plot(range(1, EPOCHS + 1), history['train_loss'], label='Train Loss', marker='o', color='red')
plt.plot(range(1, EPOCHS + 1), history['val_loss'], label='Val Loss', marker='s', color='orange')
plt.title('Training & Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('training_curves.png', dpi=300)
plt.show()

# Generate Classification Report on Validation Set
model.load_state_dict(torch.load(best_model_path))
model.eval()

all_preds, all_labels = [], []
with torch.no_grad():
    for images, labels in val_loader:
        images = images.to(device)
        outputs = model(images)
        _, preds = torch.max(outputs, 1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.numpy())

clean_names = [c.replace('___', ' - ').replace('_', ' ') for c in class_names]
print("\n--- CLASSIFICATION REPORT ---")
print(classification_report(all_labels, all_preds, target_names=clean_names, digits=4))


In [ ]:
# Step 8: Build Comprehensive Agronomic Remedies & Package-of-Practices Database
import json

DISEASE_REMEDIES = {
    "Apple___Apple_scab": {
        "crop": "Apple",
        "condition": "Fungal Disease",
        "diagnosis": "Apple Scab (Venturia inaequalis)",
        "symptoms": "Olive-green to black velvety spots on leaves; fruit develops brown corky scabs.",
        "treatment": "Spray Mancozeb 75% WP @ 2.5 g/L or Difenoconazole 25% EC @ 0.5 mL/L.",
        "prevention": "Prune infected canopy twigs and clear fallen leaf litter beneath trees."
    },
    "Apple___Black_rot": {
        "crop": "Apple",
        "condition": "Fungal Disease",
        "diagnosis": "Black Rot (Botryosphaeria obtusa)",
        "symptoms": "Frog-eye leaf spots with purple margins; firm rot on fruit expanding in concentric rings.",
        "treatment": "Apply Captan 50% WP @ 2.5 g/L or Thiophanate-methyl 70% WP @ 1 g/L.",
        "prevention": "Remove dead wood, mummified fruits, and fire blight cankers during dormant pruning."
    },
    "Apple___healthy": {
        "crop": "Apple",
        "condition": "Healthy",
        "diagnosis": "Healthy Apple Foliage",
        "symptoms": "Vigorous green leaves without chlorotic lesions or necrosis.",
        "treatment": "No fungicide required. Maintain standard micronutrient foliar spray (Zinc, Boron).",
        "prevention": "Follow standard balanced N-P-K orchard nutrition."
    },
    "Corn_(maize)___Common_rust_": {
        "crop": "Corn (Maize)",
        "condition": "Fungal Disease",
        "diagnosis": "Common Rust (Puccinia sorghi)",
        "symptoms": "Golden-brown to cinnamon-brown powdery pustules on both upper and lower leaf surfaces.",
        "treatment": "Spray Azoxystrobin 18.2% + Difenoconazole 11.4% SC @ 1 mL/L.",
        "prevention": "Plant certified rust-resistant maize hybrids; avoid excessive high-density planting."
    },
    "Corn_(maize)___Northern_Leaf_Blight": {
        "crop": "Corn (Maize)",
        "condition": "Fungal Disease",
        "diagnosis": "Northern Corn Leaf Blight (Exserohilum turcicum)",
        "symptoms": "Long elliptical, cigar-shaped grayish-green to tan lesions parallel to leaf veins.",
        "treatment": "Apply Propiconazole 25% EC @ 1 mL/L or Mancozeb 75% WP @ 2.5 g/L.",
        "prevention": "Deep summer ploughing to bury crop residue; 2-year crop rotation with non-host crops."
    },
    "Corn_(maize)___healthy": {
        "crop": "Corn (Maize)",
        "condition": "Healthy",
        "diagnosis": "Healthy Maize Foliage",
        "symptoms": "Dark green, robust leaves with intact parallel venation.",
        "treatment": "No chemical intervention needed. Ensure split Nitrogen application at knee-high stage.",
        "prevention": "Maintain timely weed control and optimum field drainage."
    },
    "Grape___Black_rot": {
        "crop": "Grape",
        "condition": "Fungal Disease",
        "diagnosis": "Grape Black Rot (Guignardia bidwellii)",
        "symptoms": "Reddish-brown circular leaf spots with tiny black pycnidia; shriveled black mummified berries.",
        "treatment": "Spray Myclobutanil 10% WP @ 1 g/L or Mancozeb @ 2 g/L before bloom and fruit set.",
        "prevention": "Maintain vine canopy training (Bower/Y-trellis) for maximum sunlight and aeration."
    },
    "Grape___Esca_(Black_Measles)": {
        "crop": "Grape",
        "condition": "Fungal Complex",
        "diagnosis": "Esca / Black Measles",
        "symptoms": "Tiger-stripe interveinal chlorosis and necrosis on leaves; dark spotting on berry skin.",
        "treatment": "Protect pruning wounds with Copper Oxychloride 50% WP paste; no single systemic cure.",
        "prevention": "Disinfect pruning shears between vines; remove and burn severely declining vines."
    },
    "Grape___healthy": {
        "crop": "Grape",
        "condition": "Healthy",
        "diagnosis": "Healthy Grape Vine Foliage",
        "symptoms": "Vibrant lobed leaves without spotting or margin burn.",
        "treatment": "No fungicide needed. Apply prophylactic organic Trichoderma viride drench.",
        "prevention": "Standard shoot thinning and balanced potassium fertigation."
    },
    "Potato___Early_blight": {
        "crop": "Potato",
        "condition": "Fungal Disease",
        "diagnosis": "Early Blight (Alternaria solani)",
        "symptoms": "Concentric target-board ring lesions on older leaves, surrounded by yellow chlorotic halo.",
        "treatment": "Spray Chlorothalonil 75% WP @ 2 g/L or Azoxystrobin 23% SC @ 1 mL/L.",
        "prevention": "Avoid overhead sprinkler irrigation; practice 3-year solanaceous crop rotation."
    },
    "Potato___Late_blight": {
        "crop": "Potato",
        "condition": "Oomycete / Water Mold",
        "diagnosis": "Potato Late Blight (Phytophthora infestans)",
        "symptoms": "Water-soaked dark lesions rapidly expanding in cool humid weather with white mildew under leaf.",
        "treatment": "Immediately spray Metalaxyl-M + Mancozeb @ 2.5 g/L or Dimethomorph 50% WP @ 1 g/L.",
        "prevention": "Plant certified disease-free seed tubers; ensure proper earthing-up to shield tubers."
    },
    "Potato___healthy": {
        "crop": "Potato",
        "condition": "Healthy",
        "diagnosis": "Healthy Potato Crop",
        "symptoms": "Full green compound leaves with upright stem vigor.",
        "treatment": "No fungicide application needed. Maintain earthing up and soil moisture.",
        "prevention": "Scout field weekly during high humidity weather."
    },
    "Tomato___Bacterial_spot": {
        "crop": "Tomato",
        "condition": "Bacterial Disease",
        "diagnosis": "Tomato Bacterial Spot (Xanthomonas spp.)",
        "symptoms": "Small angular dark water-soaked leaf spots; margins turn yellow with necrotic tears.",
        "treatment": "Spray Copper Hydroxide 53.8% DF @ 2 g/L mixed with Streptocycline @ 0.1 g/L.",
        "prevention": "Use hot-water treated certified seeds; avoid handling plants while foliage is wet."
    },
    "Tomato___Early_blight": {
        "crop": "Tomato",
        "condition": "Fungal Disease",
        "diagnosis": "Tomato Early Blight (Alternaria solani)",
        "symptoms": "Target-board concentric ring spots on lower foliage; premature defoliation.",
        "treatment": "Spray Mancozeb 75% WP @ 2.5 g/L or Tebuconazole 25.9% EC @ 1 mL/L.",
        "prevention": "Mulch around base to prevent soil splash; stake tomato vines off the ground."
    },
    "Tomato___Late_blight": {
        "crop": "Tomato",
        "condition": "Oomycete Disease",
        "diagnosis": "Tomato Late Blight (Phytophthora infestans)",
        "symptoms": "Large irregular dark greasy water-soaked lesions on leaves and stems with white fungal mold.",
        "treatment": "Spray Cymoxanil 8% + Mancozeb 64% WP @ 2 g/L or Fosetyl-Al 80% WP @ 2.5 g/L.",
        "prevention": "Ensure wide spacing (60x45 cm) for canopy aeration; destroy infected plants immediately."
    },
    "Tomato___Leaf_Mold": {
        "crop": "Tomato",
        "condition": "Fungal Disease",
        "diagnosis": "Tomato Leaf Mold (Passalora fulva)",
        "symptoms": "Pale green or yellowish spots on upper leaf surface with olive-green velvety mold underneath.",
        "treatment": "Spray Copper Oxychloride 50% WP @ 2.5 g/L or Difenoconazole @ 0.5 mL/L.",
        "prevention": "Reduce humidity below 85% in polyhouses/fields; prune lower suckers for ventilation."
    },
    "Tomato___Septoria_leaf_spot": {
        "crop": "Tomato",
        "condition": "Fungal Disease",
        "diagnosis": "Septoria Leaf Spot (Septoria lycopersici)",
        "symptoms": "Numerous circular small spots (1-3mm) with gray centers and dark brown margins.",
        "treatment": "Spray Chlorothalonil 75% WP @ 2 g/L or Mancozeb @ 2.5 g/L at first symptom appearance.",
        "prevention": "Rotate crops; remove volunteer nightshade family weeds around field borders."
    },
    "Tomato___Target_Spot": {
        "crop": "Tomato",
        "condition": "Fungal Disease",
        "diagnosis": "Tomato Target Spot (Corynespora cassiicola)",
        "symptoms": "Brown lesions with pinpoint centers and concentric zones on leaves and stems.",
        "treatment": "Apply Azoxystrobin 23% SC @ 1 mL/L or Pyraclostrobin 20% WG @ 1 g/L.",
        "prevention": "Ensure balanced nitrogen fertilization; avoid water stress during fruiting."
    },
    "Tomato___Yellow_Leaf_Curl_Virus": {
        "crop": "Tomato",
        "condition": "Viral Disease (Whitefly-borne)",
        "diagnosis": "Tomato Yellow Leaf Curl Virus (TYLCV)",
        "symptoms": "Severe upward curling and cupping of leaflets, marginal chlorosis, stunted bushy plant growth.",
        "treatment": "Virus cannot be cured; control whitefly vector: Spray Diafenthiuron 50% WP @ 1.2 g/L or Neem Oil @ 3 mL/L.",
        "prevention": "Install Yellow Sticky Traps @ 15 traps/acre; rogue out early infected symptomatic plants."
    },
    "Tomato___healthy": {
        "crop": "Tomato",
        "condition": "Healthy",
        "diagnosis": "Healthy Tomato Plant",
        "symptoms": "Vibrant green serrated foliage without spotting, curling, or mildew.",
        "treatment": "No pesticide required. Apply bio-fertilizer (Pseudomonas fluorescens) as soil drench.",
        "prevention": "Maintain consistent drip irrigation and calcium feeding to prevent blossom end rot."
    }
}

# Save class index mapping
class_indices = {i: name for i, name in enumerate(class_names)}

with open('class_indices.json', 'w') as f:
    json.dump(class_indices, f, indent=2)

with open('disease_remedies.json', 'w') as f:
    json.dump(DISEASE_REMEDIES, f, indent=2)

print(f"✅ Successfully created class_indices.json and disease_remedies.json for {len(DISEASE_REMEDIES)} classes!")


In [ ]:
# Step 9: Export Model to ONNX and Create Download Package
import zipfile
from google.colab import files

# Export to ONNX for runtime-agnostic inference
dummy_input = torch.randn(1, 3, 224, 224, device=device)
onnx_path = "plant_disease_model.onnx"

print("Exporting model to ONNX format...")
torch.onnx.export(
    model,
    dummy_input,
    onnx_path,
    input_names=['image'],
    output_names=['probabilities'],
    dynamic_axes={'image': {0: 'batch_size'}, 'probabilities': {0: 'batch_size'}},
    opset_version=14
)
print("✅ ONNX Export complete!")

# Create zip package
zip_filename = "krishimitra_disease_model.zip"
with zipfile.ZipFile(zip_filename, 'w') as zipf:
    zipf.write(best_model_path)
    zipf.write(onnx_path)
    zipf.write('class_indices.json')
    zipf.write('disease_remedies.json')
    zipf.write('training_curves.png')

print(f"\n📦 Package created: {zip_filename} ({os.path.getsize(zip_filename) / (1024*1024):.2f} MB)")
print("\n⬇️ Triggering automatic browser download...")
files.download(zip_filename)
print("✅ Download started! Unzip and move files into your local 'krishimitra/model/' folder.")
